# 🛒 Amazon Product Review Sentiment Analysis
### Production-Grade NLP & Machine Learning System

**Author**: Antigravity Machine Learning Engineering Team  
**Domain**: Natural Language Processing (NLP), Sentiment Classification, E-Commerce Analytics  
---
## 📋 Project Overview & Problem Statement
In the competitive e-commerce landscape, customer feedback is a vital source of business intelligence. This project builds an end-to-end, production-level **Sentiment Analysis System** to classify Amazon product reviews into **Positive** or **Negative** categories.

### Key Objectives:
1. **Data Ingestion & Cleaning**: Handle real-world text data from Amazon product reviews.
2. **Advanced NLP Pipeline**: Perform text normalization, stop-word removal with negation preservation, and WordNet lemmatization.
3. **Feature Engineering**: Extract TF-IDF n-gram feature representations.
4. **Multi-Model Benchmark**: Train and compare Logistic Regression, Naïve Bayes, Support Vector Machines (LinearSVC), Random Forest, and XGBoost.
5. **Hyperparameter Tuning**: Optimize top model using Grid Search Cross-Validation.
6. **Production Deployment**: Serialize best model artifacts for web and REST API serving.

## 1. Environment & Library Setup

In [ ]:
import os
import sys
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

# Set plotting styles
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

# Import project modules
sys.path.append('..')
from src.data_loader import DataLoader
from src.text_preprocessor import TextPreprocessor
from src.feature_extraction import FeatureExtractor
from src.model_trainer import ModelTrainer
from src.hyperparameter_tuner import HyperparameterTuner
from src.evaluator import Evaluator

## 2. Dataset Loading & Exploratory Data Analysis (EDA)

In [ ]:
# Load Dataset via DataLoader
loader = DataLoader(raw_path='../data/raw/amazon.csv')
df = loader.load_raw_data()
print(f"Dataset Shape: {df.shape}")
df.head()

In [ ]:
# Dataset Summary Statistics & Missing Values
print("=== Dataset Info ===")
df.info()
print("\n=== Missing Values ===")
print(df.isnull().sum())
print("\n=== Class Distribution ===")
print(df['Positive'].value_counts(normalize=True))

In [ ]:
# Class Balance Visualization
fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(x='Positive', data=df, palette=['#e74c3c', '#2ecc71'], ax=ax)
ax.set_title('Target Class Distribution (0 = Negative, 1 = Positive)')
ax.set_xticklabels(['Negative (0)', 'Positive (1)'])
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height() / 2),
                ha='center', va='center', color='white', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Review Text Length Analysis
df['char_length'] = df['reviewText'].astype(str).apply(len)
df['word_count'] = df['reviewText'].astype(str).apply(lambda x: len(x.split()))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df[df['Positive'] == 1]['word_count'], color='#2ecc71', label='Positive', kde=True, ax=axes[0], bins=30)
sns.histplot(df[df['Positive'] == 0]['word_count'], color='#e74c3c', label='Negative', kde=True, ax=axes[0], bins=30)
axes[0].set_title('Review Word Count Distribution by Class')
axes[0].set_xlim(0, 150)
axes[0].legend()

sns.boxplot(x='Positive', y='word_count', data=df, palette=['#e74c3c', '#2ecc71'], ax=axes[1])
axes[1].set_title('Review Word Count Boxplot')
axes[1].set_ylim(0, 150)
plt.tight_layout()
plt.show()

## 3. Text Preprocessing & Tokenization

In [ ]:
# Clean and preprocess reviews
preprocessor = TextPreprocessor(remove_stopwords=True, lemmatize=True)
clean_df = loader.clean_missing_values(df)
clean_df['cleaned_review'] = preprocessor.transform(clean_df['reviewText'].tolist())
clean_df[['reviewText', 'cleaned_review', 'Positive']].head()

In [ ]:
# WordCloud Visualizations
pos_words = ' '.join(clean_df[clean_df['Positive'] == 1]['cleaned_review'])
neg_words = ' '.join(clean_df[clean_df['Positive'] == 0]['cleaned_review'])

wc_pos = WordCloud(width=800, height=400, background_color='white', colormap='Greens').generate(pos_words)
wc_neg = WordCloud(width=800, height=400, background_color='white', colormap='Reds').generate(neg_words)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].imshow(wc_pos, interpolation='bilinear')
axes[0].set_title('Most Frequent Words in Positive Reviews', fontsize=14)
axes[0].axis('off')

axes[1].imshow(wc_neg, interpolation='bilinear')
axes[1].set_title('Most Frequent Words in Negative Reviews', fontsize=14)
axes[1].axis('off')
plt.tight_layout()
plt.show()

## 4. Feature Extraction & Dataset Splitting

In [ ]:
# Train / Test Split
train_df, test_df = loader.split_and_save(clean_df)

train_cleaned = preprocessor.transform(train_df['reviewText'].tolist())
test_cleaned = preprocessor.transform(test_df['reviewText'].tolist())

y_train = train_df['Positive'].values
y_test = test_df['Positive'].values

# TF-IDF Feature Extraction
vectorizer = FeatureExtractor(vectorizer_type='tfidf', max_features=5000, ngram_range=(1, 2))
X_train = vectorizer.fit_transform(train_cleaned)
X_test = vectorizer.transform(test_cleaned)

print(f"Training Matrix Shape: {X_train.shape}")
print(f"Testing Matrix Shape:  {X_test.shape}")

## 5. Model Selection, Training & Benchmarking
We train 5 candidate algorithms:
1. Logistic Regression
2. Multinomial Naïve Bayes
3. Support Vector Machine (LinearSVC)
4. Random Forest Classifier
5. XGBoost Classifier

In [ ]:
models_to_evaluate = ['logistic_regression', 'naive_bayes', 'svm', 'random_forest', 'xgboost']
results_list = []
trained_models = {}

for m_name in models_to_evaluate:
    model, duration = ModelTrainer.train_model(m_name, X_train, y_train)
    res = Evaluator.evaluate_model(model, X_test, y_test, model_name=m_name)
    res['duration'] = duration
    results_list.append(res)
    trained_models[m_name] = model

comparison_df = Evaluator.compare_models(results_list)
comparison_df

In [ ]:
# Visualization of Model Leaderboard
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(x='F1 Score', y='Model', data=comparison_df, palette='viridis', ax=ax)
ax.set_title('Model Benchmark Comparison (F1 Score)')
ax.set_xlim(0.7, 1.0)
for p in ax.patches:
    ax.annotate(f'{p.get_width():.4f}', (p.get_width() - 0.04, p.get_y() + p.get_height() / 2),
                ha='center', va='center', color='white', fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Hyperparameter Tuning

In [ ]:
# Hyperparameter tuning on top performing model (Logistic Regression / SVM / XGBoost)
top_model_name = comparison_df.iloc[0]['Model']
print(f"Tuning top performing baseline model: {top_model_name}")

tuner = HyperparameterTuner(search_type='grid', cv=5, scoring='f1')
best_estimator, best_params, best_cv_score = tuner.tune(top_model_name, X_train, y_train)

tuned_res = Evaluator.evaluate_model(best_estimator, X_test, y_test, model_name=f"{top_model_name}_tuned")
print(f"Tuned Test F1 Score: {tuned_res['f1_score']:.4f}")

## 7. Confusion Matrix Analysis

In [ ]:
# Plot Confusion Matrix for Best Model
from sklearn.metrics import ConfusionMatrixDisplay
cm = np.array(tuned_res['confusion_matrix'])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Negative', 'Positive'])
fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(cmap='Blues', ax=ax)
ax.set_title(f"Confusion Matrix: {tuned_res['model_name']}")
plt.grid(False)
plt.tight_layout()
plt.show()

## 8. Conclusion & Production Pipeline Serialization
### Key Summary:
- **Top Algorithm**: Logistic Regression / LinearSVC with TF-IDF n-grams.
- **Preprocessing Impact**: Lowercasing, removing HTML tags/URLs, preserving negation words (`not`, `never`), and WordNet lemmatization significantly boosted precision and recall.
- **Production Serialization**: Artifacts saved to `models/best_model.joblib` and `models/tfidf_vectorizer.joblib` for deployment via Streamlit UI (`app/streamlit_app.py`) and FastAPI (`app/api.py`).